# 📖 Notebook 2: Wide Rows and Clustering Columns

In Notebook 1, we learned that the **partition key** decides which node stores your data.
Now we'll explore the other half of the primary key: **clustering columns**.

Clustering columns determine how rows are **sorted within a partition**. This gives you
powerful, efficient queries that read ranges of sorted data from a single partition — what
Cassandra calls a "wide row."

## Learning Objectives

By the end of this notebook, you'll understand:
- What clustering columns are and how they sort data
- How to use ASC and DESC ordering
- What a "wide row" is and why it's powerful
- How to do range queries within a partition
- The Ticketmaster example: denormalization and multi-table design

## 🛠️ Setup

Make sure the cluster is running:

```bash
cd 03-technologies/databases/cassandra
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
from cassandra import ConsistencyLevel
from cassandra.cluster import Cluster
from tabulate import tabulate
import random

# max_schema_agreement_wait: after a CREATE TABLE, the schema has to reach
# every node before another node will answer a read against that table.
# The driver default (10s) is not enough on a freshly-formed 3-node ring,
# and the symptom is a confusing
#   ReadFailure ... INCOMPATIBLE_SCHEMA
# on the very first SELECT rather than on the DDL that caused it.
cluster = Cluster(["localhost"], port=9042, max_schema_agreement_wait=60)
session = cluster.connect()

# --- Make DDL deterministic on a multi-node ring -----------------------------
# Cassandra propagates schema changes by gossip. On this 3-node cluster a
# CREATE TABLE followed immediately by a read or write can reach a replica that
# has not seen the new schema yet, and the coordinator answers with a confusing
#   ReadFailure / WriteFailure ... INCOMPATIBLE_SCHEMA
# pointing at a random node. Waiting for every node to agree after each DDL
# statement removes that race, so these notebooks behave the same every run.
# LWT (Paxos) and QUORUM operations on a small local ring can exceed the
# driver default of 10s while the cluster is still warming up.
session.default_timeout = 30
_execute = session.execute


def _execute_and_settle(query, *args, **kwargs):
    """session.execute, hardened against schema gossip still settling.

    Two things happen here:

    1. After DDL we wait for every node to agree on the new schema.
    2. If a query still comes back INCOMPATIBLE_SCHEMA, we retry it. On a ring
       whose third node finished bootstrapping seconds ago, `wait_for_schema_
       agreement` can return while a replica is not yet serving the new table,
       and the coordinator answers Read/WriteFailure naming that node. It is
       transient, and retrying is exactly what a production client does with a
       transient coordinator failure.
    """
    import time as _time

    text = query if isinstance(query, str) else getattr(query, "query_string", "")
    is_ddl = text.strip().upper().startswith(("CREATE", "ALTER", "DROP", "TRUNCATE"))

    last = None
    for attempt in range(8):
        try:
            result = _execute(query, *args, **kwargs)
            if is_ddl:
                cluster.control_connection.wait_for_schema_agreement(wait_time=60)
            return result
        except Exception as exc:
            # INCOMPATIBLE_SCHEMA: a replica has not caught up with the new schema.
            # timed out / CAS operation timed out: a replica was too slow to answer.
            # Both are transient on a ring that has just booted; a real client
            # retries them too. Anything else is a genuine error -- re-raise it.
            transient = ("INCOMPATIBLE_SCHEMA" in str(exc)) or ("timed out" in str(exc).lower())
            if not transient:
                raise
            last = exc
            _time.sleep(2 * (attempt + 1))
    raise RuntimeError(
        f"Schema never settled after 8 retries. Last error: {last}\n"
        "Check `docker compose exec cassandra-node1 nodetool describecluster` -- "
        "all three nodes should report a single schema version."
    ) from last


session.execute = _execute_and_settle


def wait_for_cluster_ready(timeout=180.0):
    """Block until the ring can actually create a table and write to it at ALL.

    `docker compose up --wait` returns as soon as every node reports UN, but a
    node that has only just finished bootstrapping will still reject queries
    for a short while with INCOMPATIBLE_SCHEMA. Rather than sprinkle sleeps
    around, we prove readiness once: create a throwaway RF=3 table, write to it
    at consistency ALL (so every replica must answer), then drop it.
    """
    import time as _time

    from cassandra.query import SimpleStatement as _SimpleStatement
    from cassandra import ConsistencyLevel as _CL

    deadline = _time.time() + timeout
    last = None
    while _time.time() < deadline:
        try:
            _execute(
                "CREATE KEYSPACE IF NOT EXISTS readiness_probe WITH replication = "
                "{'class': 'SimpleStrategy', 'replication_factor': 3}"
            )
            cluster.control_connection.wait_for_schema_agreement(wait_time=60)
            _execute("CREATE TABLE IF NOT EXISTS readiness_probe.ping (id int PRIMARY KEY)")
            cluster.control_connection.wait_for_schema_agreement(wait_time=60)
            _execute(_SimpleStatement(
                "INSERT INTO readiness_probe.ping (id) VALUES (1)",
                consistency_level=_CL.ALL,
            ))
            _execute("DROP KEYSPACE readiness_probe")
            cluster.control_connection.wait_for_schema_agreement(wait_time=60)
            return
        except Exception as exc:  # replica still settling -- back off and retry
            last = exc
            _time.sleep(3)
    raise TimeoutError(
        f"Cassandra ring never became ready within {timeout}s. Last error: {last}\n"
        "Check `docker compose exec cassandra-node1 nodetool status` -- you need 3 UN nodes."
    )


wait_for_cluster_ready()
print("Ring ready: all 3 replicas accept schema changes and ALL-consistency writes")
# -----------------------------------------------------------------------------

# Create and use our demo keyspace
session.execute("""
    CREATE KEYSPACE IF NOT EXISTS demo
    WITH REPLICATION = { 'class': 'SimpleStrategy', 'replication_factor': 3 }
""")
session.set_keyspace('demo')

# --- Read-your-writes on a 3-node ring ---------------------------------------
# The driver's default consistency level is LOCAL_ONE. With replication_factor
# 3 that means a write can be acknowledged by a single replica while the very
# next read is served by a different replica that has not received it yet -- so
# a read-after-write silently returns None and the cell dies on an attribute of
# NoneType. Nothing in this notebook is about that trade-off, so we take the
# safe side: QUORUM on both reads and writes gives R + W > N, which is exactly
# what makes read-your-writes hold. Notebook 3 sets a consistency level per
# statement to demonstrate the trade-off deliberately, and a per-statement
# level always overrides this default.
session.default_consistency_level = ConsistencyLevel.QUORUM


print(f"Connected to: {cluster.metadata.cluster_name}")

## Step 1: Understanding Clustering Columns

A Cassandra primary key has two parts:

```
PRIMARY KEY (partition_key, clustering_col1, clustering_col2, ...)
                  │                         │
                  │                         └── Determines SORT ORDER within the partition
                  └── Determines WHICH NODE stores the data
```

Think of it like a filing cabinet:
- **Partition key** = which drawer to open
- **Clustering columns** = how files are sorted inside that drawer

Let's create a table that tracks sensor readings over time.

In [ ]:
# A sensor readings table with a clustering column for time ordering
session.execute("""
    CREATE TABLE IF NOT EXISTS sensor_readings (
        sensor_id text,
        reading_time timestamp,
        temperature float,
        humidity float,
        PRIMARY KEY (sensor_id, reading_time)
    ) WITH CLUSTERING ORDER BY (reading_time DESC)
""")

print("Table: sensor_readings")
print("Partition key:  sensor_id     → all readings for one sensor are together")
print("Clustering key: reading_time  → readings sorted newest-first")
print("\nThis is a 'wide row' — one partition holds MANY rows, sorted by time.")

> 💡 **Wide rows have limits.** A partition can technically hold ~2 billion cells, but keep partitions under ~100 MB and ~100,000 rows for healthy read performance. If a partition can grow unbounded (e.g., a forever-growing sensor), bucket the partition key by time (see Notebook 1's Discord example).

In [ ]:
from datetime import datetime, timedelta

# Insert readings for two sensors over the past 24 hours
insert_stmt = session.prepare("""
    INSERT INTO sensor_readings (sensor_id, reading_time, temperature, humidity)
    VALUES (?, ?, ?, ?)
""")

base_time = datetime(2024, 6, 15, 0, 0, 0)

for sensor in ['sensor-A', 'sensor-B']:
    for hour in range(24):
        reading_time = base_time + timedelta(hours=hour)
        temp = round(20.0 + random.uniform(-5, 10) + (hour * 0.3), 1)
        humidity = round(50.0 + random.uniform(-10, 10), 1)
        session.execute(insert_stmt, (sensor, reading_time, temp, humidity))

print("Inserted 48 readings (24 per sensor)")

In [ ]:
# Wide row query: get all readings for sensor-A — they come back sorted by time (DESC)
rows = session.execute("""
    SELECT sensor_id, reading_time, temperature, humidity
    FROM sensor_readings
    WHERE sensor_id = 'sensor-A'
    LIMIT 5
""")

print("Latest 5 readings for sensor-A (sorted newest → oldest):")
table_data = [[r.sensor_id, r.reading_time, r.temperature, r.humidity] for r in rows]
print(tabulate(table_data, headers=["sensor", "time", "temp (°C)", "humidity (%)"], tablefmt="grid"))
print("\n✅ No ORDER BY needed — clustering order handles it automatically!")

## Step 2: Range Queries Within a Partition

Because data is sorted by the clustering column, Cassandra can do efficient **range queries**.
This is one of the most powerful features of wide rows.

For example: "Give me all readings between 6 AM and noon."

In [ ]:
# Range query: readings between 6 AM and noon for sensor-A
rows = session.execute("""
    SELECT reading_time, temperature, humidity
    FROM sensor_readings
    WHERE sensor_id = 'sensor-A'
      AND reading_time >= '2024-06-15 06:00:00'
      AND reading_time <= '2024-06-15 12:00:00'
""")

print("Readings from 6 AM to noon:")
table_data = [[r.reading_time, r.temperature, r.humidity] for r in rows]
print(tabulate(table_data, headers=["time", "temp (°C)", "humidity (%)"], tablefmt="grid"))
print("\n✅ Range query on clustering column — efficient single-partition read!")

## Step 3: Multiple Clustering Columns

You can have more than one clustering column. They form a **nested sort**:
first by the first clustering column, then by the second within each value of the first, etc.

Let's model an e-commerce order history where we want to browse orders by category then price.

In [ ]:
# Table with two clustering columns: category (ASC) then price (DESC)
session.execute("""
    CREATE TABLE IF NOT EXISTS user_purchases (
        user_id bigint,
        category text,
        price decimal,
        product_name text,
        purchased_at timestamp,
        PRIMARY KEY (user_id, category, price)
    ) WITH CLUSTERING ORDER BY (category ASC, price DESC)
""")

print("Partition key:  user_id")
print("Clustering keys: category ASC, price DESC")
print("\nData is sorted first by category (A→Z), then by price (highest first) within each category.")

In [ ]:
from decimal import Decimal

# Insert sample purchases for user 1
insert_stmt = session.prepare("""
    INSERT INTO user_purchases (user_id, category, price, product_name, purchased_at)
    VALUES (?, ?, ?, ?, ?)
""")

purchases = [
    (1, 'Books', Decimal('29.99'), 'System Design Interview', datetime(2024, 3, 1)),
    (1, 'Books', Decimal('15.99'), 'Clean Code', datetime(2024, 3, 5)),
    (1, 'Books', Decimal('9.99'), 'The Pragmatic Programmer', datetime(2024, 4, 1)),
    (1, 'Electronics', Decimal('999.99'), 'Laptop', datetime(2024, 2, 1)),
    (1, 'Electronics', Decimal('249.99'), 'Mechanical Keyboard', datetime(2024, 2, 15)),
    (1, 'Electronics', Decimal('79.99'), 'Mouse', datetime(2024, 3, 1)),
    (1, 'Clothing', Decimal('59.99'), 'Hoodie', datetime(2024, 1, 10)),
    (1, 'Clothing', Decimal('24.99'), 'T-Shirt', datetime(2024, 1, 15)),
]

for p in purchases:
    session.execute(insert_stmt, p)

print(f"Inserted {len(purchases)} purchases")

In [ ]:
# Query all purchases for user 1 — notice the sort order
rows = session.execute("SELECT category, price, product_name FROM user_purchases WHERE user_id = 1")

print("All purchases for user 1 (sorted by category ASC, price DESC):")
table_data = [[r.category, float(r.price), r.product_name] for r in rows]
print(tabulate(table_data, headers=["category", "price", "product"], tablefmt="grid"))
print("\n↑ Notice: Books A→Z, then within each category, most expensive first!")

In [ ]:
# Range query: only Electronics priced $100+
rows = session.execute("""
    SELECT category, price, product_name
    FROM user_purchases
    WHERE user_id = 1
      AND category = 'Electronics'
      AND price >= 100
""")

print("Electronics $100+:")
table_data = [[r.category, float(r.price), r.product_name] for r in rows]
print(tabulate(table_data, headers=["category", "price", "product"], tablefmt="grid"))

## Step 4: The Ticketmaster Example — Denormalization

Let's apply what we've learned to a real-world system: Ticketmaster's ticket browsing UI.

### The Access Patterns

Ticketmaster has two main views:
1. **Venue map** — shows all sections with summary info (ticket count, lowest price)
2. **Section detail** — shows individual seats when a user clicks a section

In a relational database, you'd have one `tickets` table and write different queries.
In Cassandra, we create **one table per access pattern** and duplicate (denormalize) data.

In [ ]:
# Table 1: Individual tickets — serves the "section detail" view
session.execute("""
    CREATE TABLE IF NOT EXISTS tickets (
        event_id bigint,
        section_id bigint,
        seat_id bigint,
        price bigint,
        is_available boolean,
        PRIMARY KEY ((event_id, section_id), seat_id)
    )
""")

# Table 2: Section summaries — serves the "venue map" view
session.execute("""
    CREATE TABLE IF NOT EXISTS event_sections (
        event_id bigint,
        section_id bigint,
        num_tickets bigint,
        price_floor bigint,
        PRIMARY KEY (event_id, section_id)
    )
""")

print("Created two tables:")
print("")
print("tickets:        partition = (event_id, section_id)  → one partition per section")
print("event_sections: partition = event_id                → all sections for an event together")
print("")
print("This is DENORMALIZATION — the same data exists in two places, optimized for different queries.")

In [ ]:
# Insert ticket data for a concert with 3 sections
insert_ticket = session.prepare("""
    INSERT INTO tickets (event_id, section_id, seat_id, price, is_available)
    VALUES (?, ?, ?, ?, ?)
""")

insert_section = session.prepare("""
    INSERT INTO event_sections (event_id, section_id, num_tickets, price_floor)
    VALUES (?, ?, ?, ?)
""")

event_id = 1001
sections = {
    1: {'name': 'Floor', 'price_range': (150, 300), 'seats': 20},
    2: {'name': 'Lower Bowl', 'price_range': (75, 150), 'seats': 30},
    3: {'name': 'Upper Bowl', 'price_range': (25, 75), 'seats': 40},
}

for section_id, info in sections.items():
    min_price = info['price_range'][1]  # track lowest price
    available_count = 0

    for seat in range(1, info['seats'] + 1):
        price = random.randint(*info['price_range'])
        available = random.random() > 0.2  # 80% available
        session.execute(insert_ticket, (event_id, section_id, seat, price, available))
        if available:
            available_count += 1
            min_price = min(min_price, price)

    # Denormalized summary for the venue map view
    session.execute(insert_section, (event_id, section_id, available_count, min_price))

print(f"Inserted tickets for event {event_id} with {len(sections)} sections")

In [ ]:
# Query 1: Venue map view — all sections for the event (single partition read)
rows = session.execute(f"""
    SELECT section_id, num_tickets, price_floor
    FROM event_sections
    WHERE event_id = {event_id}
""")

section_names = {1: 'Floor', 2: 'Lower Bowl', 3: 'Upper Bowl'}
print("🗺️ Venue Map View (single partition read):")
table_data = [[section_names.get(r.section_id, r.section_id), r.num_tickets, f"${r.price_floor}+"] for r in rows]
print(tabulate(table_data, headers=["Section", "Available", "From"], tablefmt="grid"))

In [ ]:
# Query 2: Section detail view — individual seats (single partition read)
section_to_view = 1  # Floor section
rows = session.execute(f"""
    SELECT seat_id, price, is_available
    FROM tickets
    WHERE event_id = {event_id} AND section_id = {section_to_view}
""")

print(f"🪑 Floor Section Detail (single partition read):")
table_data = [[r.seat_id, f"${r.price}", "✅" if r.is_available else "❌"] for r in rows]
print(tabulate(table_data, headers=["Seat", "Price", "Available"], tablefmt="grid"))
print("\nBoth queries read from a SINGLE partition — fast and efficient!")

## Step 5: Clustering Column Restrictions

There's an important rule with multiple clustering columns:

> **You must filter clustering columns left-to-right.** You cannot skip a clustering column.

This is because data is sorted hierarchically. Skipping a level would require scanning all values
at that level — which defeats the purpose of the sort.

In [ ]:
# WORKS: Filter by first clustering column (category)
rows = session.execute("""
    SELECT category, price, product_name
    FROM user_purchases
    WHERE user_id = 1 AND category = 'Books'
""")
print("✅ Filter by category (first clustering col):")
for r in rows:
    print(f"   {r.category} - ${r.price} - {r.product_name}")

print()

# FAILS: Try to skip category and filter only by price
try:
    rows = session.execute("""
        SELECT category, price, product_name
        FROM user_purchases
        WHERE user_id = 1 AND price > 50
    """)
except Exception as e:
    print(f"❌ Cannot skip clustering columns: {e}")
    print("\nYou must filter 'category' before you can filter 'price'.")
    print("This is because data is sorted by category FIRST, then price within each category.")

## 🧠 Key Takeaways

1. **Clustering columns sort data within a partition.** The sort is physical — data is stored in this order on disk.

2. **Wide rows = many rows in one partition.** This is Cassandra's superpower for time-series and ordered data.

3. **Range queries on clustering columns are fast** because data is pre-sorted.

4. **Multiple clustering columns create nested sorts** — first by col1, then col2 within col1.

5. **Denormalization is normal.** Create separate tables optimized for each access pattern rather than trying to serve everything from one table.

6. **Filter left-to-right.** You cannot skip clustering columns in WHERE clauses.

## ➡️ Next Notebook

In Notebook 3, we'll explore **replication and consistency levels** — how Cassandra keeps copies of your data on multiple nodes and lets you choose the trade-off between speed and accuracy.

In [ ]:
cluster.shutdown()
print("Connection closed.")